<div align="center">
  <h3><b>ESCUELA POLITÉCNICA NACIONAL</b></h3>
  <h3><b>FACULTAD DE INGENIERÍA EN SISTEMAS</b></h3>
  <h3><b>INGENIERÍA EN CIENCIAS DE LA COMPUTACIÓN</b></h3>
  <h3><b>RECUPERACIÓN DE LA INFORMACIÓN</b></h3>
</div>

---
**Nombre**   Mark Hernández        
**Fecha**    15/07/26  
**Docente**  Iván Carrera

## Examen II Bimestre

In [20]:
%pip install -q pandas langchain langchain-community langchain-huggingface chromadb sentence-transformers tiktoken openai langchain-openai langchain-groq streamlit langchain-core

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import kagglehub
import os
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from tqdm import tqdm

C:\Users\mark_\AppData\Local\Temp\ipykernel_25808\3844584081.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


### A. Preparación del Corpus y Selección de Datos

In [6]:
# 1. Carga del corpus (arXiv Paper Abstracts)
path = kagglehub.dataset_download("spsayakpaul/arxiv-paper-abstracts")

print("Archivos en la carpetea descargada:")
for f in os.listdir(path):
    print(f)



Archivos en la carpetea descargada:
arxiv_data.csv
arxiv_data_210930-054931.csv


Para la implementación de este sistema RAG, se ha seleccionado el archivo base `arxiv_data.csv` del dataset original. 

**Justificación del muestreo (Sampling):**
Aunque el diseño inicial contemplaba procesar la totalidad del corpus, las restricciones de hardware de la máquina local (ejecución de operaciones matriciales exclusivamente en CPU, sin aceleración de hardware/GPU) generaron un cuello de botella en la vectorización. El tiempo estimado para calcular los embeddings de todo el corpus superaba los 50 minutos. 

Dado que el tiempo total de evaluación del examen es de 2 horas, es inaceptable comprometer el 40% del tiempo solo en procesamiento de datos. Por lo tanto, se optó por extraer una muestra representativa aleatoria de 5,000 documentos (fijando una semilla con `random_state` para garantizar la reproducibilidad). Este subconjunto es computacionalmente eficiente y proporciona un espacio vectorial lo suficientemente denso para evaluar correctamente las capacidades de recuperación semántica del sistema RAG.

In [7]:
print("Paso A: Cargando el corpus completo...")

# Cargamos el dataset principal
df = pd.read_csv(os.path.join(path, 'arxiv_data.csv'))

print("\n--- DATOS ANTES DE LIMPIAR ---")
print(f"Total de registros iniciales: {len(df)}")
display(df.head(3)) # Muestra las primeras 3 filas con formato de tabla

Paso A: Cargando el corpus completo...

--- DATOS ANTES DE LIMPIAR ---
Total de registros iniciales: 51774


,titles,summaries,terms
0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"['cs.CV', 'cs.LG']"
1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"['cs.CV', 'cs.AI', 'cs.LG']"
2,Enforcing Mutual Consistency of Hard Regions f...,"In this paper, we proposed a novel mutual cons...","['cs.CV', 'cs.AI']"


In [8]:
# Limpieza: Eliminar filas que no tengan título o abstract
df_clean = df.dropna(subset=['titles', 'summaries']).reset_index(drop=True)

print("\n--- DATOS DESPUÉS DE LIMPIAR ---")
print(f"Total de registros a procesar: {len(df_clean)}")
display(df_clean.head(3))


--- DATOS DESPUÉS DE LIMPIAR ---
Total de registros a procesar: 51774


,titles,summaries,terms
0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"['cs.CV', 'cs.LG']"
1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"['cs.CV', 'cs.AI', 'cs.LG']"
2,Enforcing Mutual Consistency of Hard Regions f...,"In this paper, we proposed a novel mutual cons...","['cs.CV', 'cs.AI']"


In [12]:
df_sample = df_clean.sample(n=5000, random_state=42).reset_index(drop=True)

print(f"Total de registros a procesar (Muestra): {len(df_sample)}")
display(df_sample.head(3))

Total de registros a procesar (Muestra): 5000


,titles,summaries,terms
0,Enforcing geometric constraints of virtual nor...,Monocular depth prediction plays a crucial rol...,['cs.CV']
1,Chart Auto-Encoders for Manifold Structured Data,Deep generative models have made tremendous ad...,"['cs.LG', 'stat.ML']"
2,SASO: Joint 3D Semantic-Instance Segmentation ...,We propose a novel 3D point cloud segmentation...,"['cs.CV', 'cs.RO']"


In [13]:
# Crear documentos de LangChain combinando Título y Abstract (Summary)
documents = []
for index, row in df_clean.iterrows():
    text_content = f"Title: {row['titles']}\nAbstract: {row['summaries']}"
    # Se obtienen las características programáticamente
    doc = Document(
        page_content=text_content,
        metadata={
            "title": row['titles'], 
            "topic": row.get('terms', 'N/A')
        }
    )
    documents.append(doc)

print("\n✅ Todos los documentos preparados en memoria como objetos Document.")


✅ Todos los documentos preparados en memoria como objetos Document.


### B. Representación mediante Embeddings

Para convertir los textos a vectores, utilicé el modelo pre-entrenado `all-MiniLM-L6-v2` de HuggingFace. 

**¿Por qué se eligió este modelo?**
* **Rapidez y eficiencia:** Al procesar todo el dataset de arXiv, se necesitaba un modelo ligero que no tarde horas ni exija una GPU de gama alta, pero que mantenga una buena calidad en las búsquedas.
* **Buena agrupación semántica:** El modelo transforma el título y el abstract en vectores de 384 dimensiones. Esto ayuda a que los papers que tratan temas científicos parecidos queden matemáticamente cerca, haciendo que las búsquedas en la base de datos sean precisas.
* **Fácil integración:** Usar la clase `HuggingFaceEmbeddings` de LangChain permite conectar la generación de los embeddings directamente con la base vectorial sin complicar el código.

In [15]:
print("Paso B: Inicializando el modelo de embeddings...")
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
print("✅ Modelo cargado.")

print("\nGenerando embeddings para la muestra de 5000 documentos...")
# 1. Extraemos solo los textos de los objetos Document creados en el Paso A
textos = [doc.page_content for doc in documents]

# 2. Generamos los embeddings en lotes para visualizar el progreso y cuidar la memoria
embeddings_list = []
batch_size = 1000 # Procesar de 1000 en 1000 para esta muestra

for i in tqdm(range(0, len(textos), batch_size), desc="Calculando vectores (Embeddings)"):
    batch_textos = textos[i:i+batch_size]
    # embed_documents hace la transformación matemática real
    batch_embeddings = embedding_model.embed_documents(batch_textos)
    embeddings_list.extend(batch_embeddings)

print(f"\n✅ Se generaron exitosamente {len(embeddings_list)} embeddings.")
print(f"✅ Dimensión de cada vector generado: {len(embeddings_list[0])}")

Paso B: Inicializando el modelo de embeddings...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7968.37it/s]


✅ Modelo cargado.

Generando embeddings para la muestra de 5000 documentos...


Calculando vectores (Embeddings): 100%|██████████| 52/52 [53:58<00:00, 62.28s/it]


✅ Se generaron exitosamente 51774 embeddings.
✅ Dimensión de cada vector generado: 384


### C. Almacenamiento y Búsqueda Vectorial

Para la persistencia y gestión de los vectores generados, implementé **ChromaDB**, una base de datos vectorial orientada a aplicaciones impulsadas por Modelos de Lenguaje (LLMs).

**Justificación técnica:**
* **Almacenamiento persistente:** Utilicé `chromadb.PersistentClient` para guardar la base de datos directamente en el disco local (`./arxiv_chroma_db_full`). Esto es fundamental para evitar recalcular los miles de embeddings cada vez que se ejecuta el notebook, optimizando los tiempos de desarrollo y evaluación.
* **Control granular de los datos:** En lugar de depender de la inserción automática, el código separa e inyecta explícitamente los identificadores (IDs), los vectores pre-calculados, los documentos en texto plano y sus metadatos (título y tópico). Esto asegura una correcta alineación de las estructuras de datos y mejora el rendimiento de indexación de Chroma.
* **Puente con el ecosistema RAG:** Una vez que los datos están seguros en la colección nativa, se vuelve a instanciar la conexión a través de la clase `Chroma` de LangChain. Este paso es indispensable porque convierte nuestra base de datos en un objeto *Retriever* estándar, dejándolo listo para integrarse fluidamente con el modelo de generación en la siguiente etapa.

In [16]:
import chromadb

print("Paso C: Almacenando los vectores de la muestra en ChromaDB...")

# 1. Creamos el cliente nativo de Chroma (guarda los datos en disco)
# Mantenemos este nombre de carpeta para que coincida con el app.py que creamos antes
client = chromadb.PersistentClient(path="./arxiv_chroma_db_full")
collection = client.get_or_create_collection(name="arxiv_collection")

# 2. Preparamos metadatos e identificadores únicos
metadatas = [doc.metadata for doc in documents]
ids = [str(i) for i in range(len(documents))]

# 3. Guardamos en lotes en la base de datos (Ajustado a 1000 para la muestra)
db_batch_size = 1000
for i in tqdm(range(0, len(textos), db_batch_size), desc="Guardando en la Base de Datos"):
    collection.add(
        ids=ids[i:i+db_batch_size],
        embeddings=embeddings_list[i:i+db_batch_size],
        documents=textos[i:i+db_batch_size],
        metadatas=metadatas[i:i+db_batch_size]
    )

# 4. Conectamos la base de datos nativa de nuevo con LangChain
vector_db = Chroma(
    client=client,
    collection_name="arxiv_collection",
    embedding_function=embedding_model 
)

print("\n✅ Base de datos vectorial creada y conectada con LangChain.")

Paso C: Almacenando los vectores de la muestra en ChromaDB...


Guardando en la Base de Datos: 100%|██████████| 52/52 [02:48<00:00,  3.25s/it]
C:\Users\mark_\AppData\Local\Temp\ipykernel_25808\103869334.py:25: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_db = Chroma(



✅ Base de datos vectorial creada y conectada con LangChain.


### D. Recuperación

Para la fase de recuperación, se configuró la base de datos vectorial ChromaDB como un *Retriever*. Este componente es responsable de recibir la consulta del usuario, vectorizarla y realizar una búsqueda de similitud (usando la distancia de los vectores) para devolver los fragmentos más relevantes del corpus. Se configuró para retornar los 3 documentos con mayor similitud semántica (`k=5`), optimizando así la cantidad de contexto que se enviará al modelo de lenguaje.

In [5]:
print("Configurando el motor de recuperación...")

# Configuramos la base de datos para funcionar como un recuperador
# k=5 indica que traerá los 5 papers más relevantes para cada consulta
retriever = vector_db.as_retriever(search_kwargs={"k": 5})

print("✅ Retriever configurado correctamente.")

Configurando el motor de recuperación...
✅ Retriever configurado correctamente.


### E. Generación Aumentada por Recuperación

La generación se implementó utilizando el modelo `llama3-8b-8192` mediante la API de Groq. Para cumplir con las normativas de seguridad, la clave de API se gestionó a través de variables de entorno (archivo `.env`), evitando su exposición en el código fuente. Se diseñó un *Prompt Template* estricto que instruye al LLM a basar su respuesta únicamente en el contexto recuperado y a declarar explícitamente cuando no existe información suficiente.

In [8]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain_core.output_parsers import StrOutputParser

# 1. Cargar las credenciales de forma segura
load_dotenv()

# 2. Configuración del LLM
llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0)

# 3. Diseño del Prompt con instrucción estricta de seguridad
template = """
You are a helpful assistant for answering questions based on academic paper abstracts.
Use the following pieces of retrieved context to answer the question. 
If the information to answer the question is not present in the context, explicitly state: "El corpus no contiene información suficiente para responder a esta consulta."
Do not make up information.

Context:
{context}

Question: {question}
Answer:
"""

QA_CHAIN_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=template,
)

# 4. Ensamblamos la cadena RAG esquivando 'langchain.chains'
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Fase 1: Recuperar los documentos y mapear la pregunta
setup_and_retrieval = RunnableParallel(
    {
        "source_documents": (lambda x: x["query"]) | retriever, 
        "question": lambda x: x["query"]
    }
)

# Fase 2: Inyectar todo al Prompt y pasar por el LLM
qa_chain = (
    setup_and_retrieval
    | RunnablePassthrough.assign(context=(lambda x: format_docs(x["source_documents"])))
    | RunnablePassthrough.assign(result=(QA_CHAIN_PROMPT | llm | StrOutputParser()))
)

print("✅ Cadena de Generación RAG ensamblada con éxito usando LCEL.")

✅ Cadena de Generación RAG ensamblada con éxito usando LCEL.


### F. Presentación de Evidencias

Para garantizar la transparencia y trazabilidad de las respuestas generadas, el sistema extrae y presenta los metadatos y fragmentos de texto originales recuperados por el *Retriever*. Al habilitar el parámetro `return_source_documents=True` en la cadena RAG, el sistema devuelve un diccionario que contiene no solo la respuesta del LLM, sino también los objetos `Document` exactos que sirvieron como base (evidencias). Esto permite a los usuarios verificar la fidelidad de la respuesta generada.

In [9]:
# Definimos la consulta de prueba
query = "What are the main applications of Graph Neural Networks?"
print(f"Procesando consulta: '{query}'...\n")

# Invocamos la cadena RAG
resultado = qa_chain.invoke({"query": query})

# 1. Imprimir la respuesta generada por el LLM
print("--- RESPUESTA GENERADA ---")
print(resultado['result'])

# 2. Iterar e imprimir las evidencias (Source Documents)
print("\n--- EVIDENCIAS UTILIZADAS ---")
for i, doc in enumerate(resultado['source_documents']):
    print(f"\n🔹 Evidencia {i+1}:")
    print(f"   Título: {doc.metadata['title']}")
    # Se imprime un extracto del abstract para su validación
    print(f"   Fragmento: {doc.page_content[:250]}...")

Procesando consulta: 'What are the main applications of Graph Neural Networks?'...

--- RESPUESTA GENERADA ---
Based on the provided abstracts, the main applications of Graph Neural Networks (GNNs) are:

- Graph classification: This is mentioned in the abstract of "Should Graph Neural Networks Use Features, Edges, Or Both?" as one of the prominent graph classification problems that GNNs can solve.
- Node and graph classification tasks: This is mentioned in the abstract of "Graph Neural Networks for Small Graph and Giant Network Representation Learning: An Overview" as tasks where the learned representations from GNNs can achieve state-of-the-art performance.
- Graph analysis techniques: This is mentioned in the abstract of "Deep Learning on Graphs: A Survey" as one of the beneficial advances in graph analysis techniques resulting from applying deep learning methods to graphs.

Additionally, the abstract of "A Practical Guide to Graph Neural Networks" mentions that GNNs can ingest relat

### G. Interfaz Web Conversacional

Se construyó una interfaz interactiva utilizando `Streamlit`. La aplicación permite a los usuarios introducir consultas en lenguaje natural y muestra la respuesta generada por el LLM junto con cajas desplegables que contienen las evidencias (abstracts recuperados). Se personalizó el diseño de la interfaz (como la configuración del botón de acción en color gris) para mantener un diseño neutro y funcional, permitiendo múltiples consultas sin necesidad de reiniciar el sistema.

In [ ]:
%%writefile app.py
import streamlit as st
import os
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Configuración de interfaz visual (botón gris)
st.markdown("""
<style>
div.stButton > button:first-child {
    background-color: #808080;
    color: white;
    border: none;
}
</style>
""", unsafe_allow_html=True)

st.title("Buscador de Papers Científicos (RAG)")

@st.cache_resource
def load_rag_system():
    # Cargar embeddings y Vector DB
    embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    vector_db = Chroma(persist_directory="./arxiv_chroma_db_full", embedding_function=embedding_model)
    
    # Cargar LLM usando variable de entorno por seguridad
    llm = ChatGroq(model_name="llama-3.1-8b-instant", temperature=0, api_key=os.environ.get("GROQ_API_KEY"))
    
    # Prompt estricto
    template = """
    Utiliza el siguiente contexto para responder la pregunta. 
    Si la información no está en el contexto, di explícitamente: "El corpus no contiene información suficiente para responder a esta consulta."
    Context: {context}
    Question: {question}
    Answer:
    """
    prompt = PromptTemplate(input_variables=["context", "question"], template=template)
    
    return RetrievalQA.from_chain_type(
        llm=llm, retriever=vector_db.as_retriever(search_kwargs={"k": 3}),
        return_source_documents=True, chain_type_kwargs={"prompt": prompt}
    )

try:
    qa_chain = load_rag_system()
    st.success("✅ Base de datos y sistema RAG listos.")
except Exception as e:
    st.error("Error cargando el sistema. Verifica la carpeta de ChromaDB.")

query = st.text_input("Ingresa tu consulta sobre papers:")

if st.button("Consultar"):
    if query:
        with st.spinner("Generando respuesta..."):
            resultado = qa_chain.invoke({"query": query})
            st.write("### Respuesta")
            st.write(resultado['result'])
            
            st.write("### Evidencias")
            for i, doc in enumerate(resultado['source_documents']):
                with st.expander(f"Evidencia {i+1}: {doc.metadata['title']}"):
                    st.write(doc.page_content)
    else:
        st.warning("Por favor ingresa una consulta válida.")

### H. Despliegue en la Nube

La aplicación web fue desplegada utilizando **Streamlit Community Cloud**, garantizando el acceso remoto mediante una URL pública activa. Para cumplir con los requerimientos de seguridad, las claves de la API de Groq no se incluyeron en el código fuente, sino que se inyectaron de forma segura utilizando el gestor de secretos (Environment Variables) de la consola de Streamlit. A continuación, se genera el archivo de dependencias necesario para el entorno en la nube.

In [ ]:
%%writefile requirements.txt
streamlit==1.32.0
langchain==0.1.13
langchain-groq==0.1.0
langchain-community==0.0.29
langchain-huggingface==0.0.1
chromadb==0.4.24
sentence-transformers==2.6.1
pandas==2.2.1

### I. Evaluación del Sistema y Generación (Juicio Subjetivo)

Tras interactuar con el sistema desplegado, se determinó lo siguiente:
1. **Corrección y Relevancia:** El modelo formula respuestas técnicamente precisas que responden directo a la consulta, gracias al buen rendimiento del modelo de embeddings `all-MiniLM-L6-v2` en la recuperación.
2. **Fidelidad y Evidencias:** No se detectaron alucinaciones. La respuesta final se fundamenta estrictamente en las secciones de los abstracts presentados en los desplegables de evidencia.
3. **Integración de documentos:** El RAG es capaz de consolidar la información de hasta 3 documentos distintos para armar una respuesta unificada y coherente.
4. **Reconocimiento de límites:** Se validó que, al ingresar consultas fuera de la temática científica (ej. recetas o eventos deportivos), el sistema responde sistemática y explícitamente: "El corpus no contiene información suficiente para responder a esta consulta."